In [4]:
import pandas as pd
import numpy as np

def reduce_mem_usage(df):
    """Safely reduce memory usage using Pandas native type checks."""
    start_mem = df.memory_usage().sum() / 1024**2
    
    for col in df.columns:
        # Check integer types safely
        if pd.api.types.is_integer_dtype(df[col]):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
                
        # Check float types safely (downcast float64 to float32)
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].astype(np.float32)
                    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Memory usage decreased from {start_mem:.2f} MB to {end_mem:.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)")
    return df

# 1. Load Data
print("Loading train_transaction...")
train_transaction = pd.read_csv('../data/raw/train_transaction.csv')
train_transaction = reduce_mem_usage(train_transaction)

print("\nLoading train_identity...")
train_identity = pd.read_csv('../data/raw/train_identity.csv')
train_identity = reduce_mem_usage(train_identity)

# 2. Left Merge on TransactionID
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
print(f"\nMerged Train Shape: {train.shape}")

# 3. Target Distribution
fraud_counts = train['isFraud'].value_counts(normalize=True) * 100
print("\nTarget Class Distribution (%):")
print(fraud_counts)

Loading train_transaction...
Memory usage decreased from 1775.15 MB to 916.30 MB (48.4% reduction)

Loading train_identity...
Memory usage decreased from 45.12 MB to 31.91 MB (29.3% reduction)

Merged Train Shape: (590540, 434)

Target Class Distribution (%):
isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64


In [5]:
import sys
sys.path.append('../')
from src.features import build_feature_matrix

# Run feature pipeline on your merged train dataframe
train_features = build_feature_matrix(train)

# Inspect the newly generated feature columns
print(train_features[['TransactionAmt', 'hour', 'amt_to_card_mean', 'amt_card_zscore']].head())

Generating time features...
Generating amount aggregation features...
   TransactionAmt  hour  amt_to_card_mean  amt_card_zscore
0            68.5     0          0.194640        -0.763675
1            29.0     0          0.123777        -0.445943
2            59.0     0          0.608150        -0.379666
3            50.0     0          0.405133        -0.380953
4            50.0     0          0.515612        -0.829466
